# 02 - Build gold churn tables
Reads the raw layer from `01_ingest_genie_telemetry` and builds the tables the app and the Genie Code skill read.

| Table | Grain | Purpose |
| --- | --- | --- |
| `gold_user_lifecycle` | user | Active / At Risk / Churned, activity and quality stats |
| `gold_monthly_user_health` | month | new, active, churned, at-risk, dormant, net change, churn rate |
| `gold_cohort_retention` | cohort x month | month-N retention curves |
| `gold_enriched_messages` | message | rephrase, latency bucket, problem flag, AI failure reason, comment sentiment (incremental) |
| `gold_churn_driver_summary` | lifecycle status | churned vs active comparison |
| `gold_failure_reasons` | space x reason x status | what goes wrong, for whom |
| `gold_space_health` | space | per-space quality and churn |
| `gold_user_risk` | user | primary churn reason + 0-100 risk score |

In [0]:
dbutils.widgets.text("catalog", "main")
dbutils.widgets.text("schema", "genie_churn")
dbutils.widgets.text("churn_days", "30")
dbutils.widgets.text("at_risk_days", "14")
dbutils.widgets.text("grant_to_principal", "")  # optional: app service principal / group that reads the gold tables

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
CHURN = int(dbutils.widgets.get("churn_days"))
AT_RISK = int(dbutils.widgets.get("at_risk_days"))
GRANTEE = dbutils.widgets.get("grant_to_principal").strip()
assert 0 < AT_RISK < CHURN, "at_risk_days must be > 0 and < churn_days"

spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")
print(f"{CATALOG}.{SCHEMA} | at risk > {AT_RISK}d | churned > {CHURN}d")

In [0]:
# One row per business user: activity, quality signals, lifecycle status as of today.
spark.sql(f"""
CREATE OR REPLACE TABLE gold_user_lifecycle AS
WITH ua AS (
  SELECT
    m.user_email,
    COALESCE(u.department, 'Unknown') AS department,
    COALESCE(u.role, 'Unknown')       AS role,
    DATE(MIN(m.created_at))           AS first_question_date,
    DATE(MAX(m.created_at))           AS last_question_date,
    COUNT(DISTINCT DATE(m.created_at)) AS active_days,
    COUNT(*)                          AS total_questions,
    COUNT(DISTINCT m.conversation_id) AS total_conversations,
    COUNT(DISTINCT m.space_id)        AS spaces_used,
    MAX_BY(m.space_id, m.created_at)  AS last_space_id,
    COUNT_IF(m.feedback_rating = 'POSITIVE') AS positive_feedback,
    COUNT_IF(m.feedback_rating = 'NEGATIVE') AS negative_feedback,
    COUNT_IF(m.status IN ('FAILED', 'CANCELLED')) AS failed_questions,
    ROUND(AVG(m.response_seconds), 1) AS avg_response_seconds,
    COUNT_IF(m.created_at >= current_date() - INTERVAL 28 DAYS) AS questions_last_28d,
    COUNT_IF(m.created_at >= current_date() - INTERVAL 56 DAYS AND m.created_at < current_date() - INTERVAL 28 DAYS) AS questions_prev_28d
  FROM raw_genie_messages m
  LEFT JOIN dim_genie_users u USING (user_email)
  WHERE m.user_email IS NOT NULL
  GROUP BY m.user_email, u.department, u.role
)
SELECT
  *,
  DATEDIFF(current_date(), last_question_date) AS days_since_last_question,
  CASE
    WHEN DATEDIFF(current_date(), last_question_date) > {CHURN}   THEN 'Churned'
    WHEN DATEDIFF(current_date(), last_question_date) > {AT_RISK} THEN 'At Risk'
    ELSE 'Active'
  END AS lifecycle_status,
  DATEDIFF(current_date(), first_question_date) <= {AT_RISK} AS is_new_user,
  ROUND(negative_feedback * 100.0 / NULLIF(positive_feedback + negative_feedback, 0), 1) AS negative_feedback_pct,
  ROUND(failed_questions * 100.0 / NULLIF(total_questions, 0), 1) AS failure_rate_pct
FROM ua
""")
display(spark.sql("SELECT lifecycle_status, COUNT(*) users, COUNT_IF(is_new_user) new_users FROM gold_user_lifecycle GROUP BY 1 ORDER BY 1"))

In [0]:
# Monthly trend. churned_users = active last month but not this month.
# at_risk_users / dormant_users = point-in-time counts at month end (or today for the current month).
spark.sql(f"""
CREATE OR REPLACE TABLE gold_monthly_user_health AS
WITH um AS (
  SELECT CAST(DATE_TRUNC('MONTH', created_at) AS DATE) AS activity_month, user_email
  FROM raw_genie_messages WHERE user_email IS NOT NULL
  GROUP BY ALL
),
months AS (
  SELECT DISTINCT activity_month AS month_start,
         LEAST(LAST_DAY(activity_month), current_date()) AS as_of
  FROM um
),
first_month AS (SELECT user_email, MIN(activity_month) AS cohort_month FROM um GROUP BY 1),
active AS (SELECT activity_month AS month_start, COUNT(DISTINCT user_email) AS active_users FROM um GROUP BY 1),
new_u  AS (SELECT cohort_month AS month_start, COUNT(*) AS new_users FROM first_month GROUP BY 1),
churned AS (
  SELECT ADD_MONTHS(prev.activity_month, 1) AS month_start, COUNT(DISTINCT prev.user_email) AS churned_users
  FROM um prev
  LEFT JOIN um cur ON prev.user_email = cur.user_email AND cur.activity_month = ADD_MONTHS(prev.activity_month, 1)
  WHERE cur.user_email IS NULL
  GROUP BY 1
),
last_seen AS (
  SELECT mo.month_start, mo.as_of, d.user_email, MAX(d.d) AS last_d
  FROM months mo
  JOIN (SELECT DISTINCT user_email, DATE(created_at) AS d FROM raw_genie_messages WHERE user_email IS NOT NULL) d
    ON d.d <= mo.as_of
  GROUP BY 1, 2, 3
),
state AS (
  SELECT month_start,
         COUNT_IF(DATEDIFF(as_of, last_d) > {AT_RISK} AND DATEDIFF(as_of, last_d) <= {CHURN}) AS at_risk_users,
         COUNT_IF(DATEDIFF(as_of, last_d) > {CHURN}) AS dormant_users,
         COUNT(*) AS cumulative_users
  FROM last_seen GROUP BY 1
)
SELECT
  a.month_start,
  a.active_users,
  COALESCE(n.new_users, 0)     AS new_users,
  COALESCE(c.churned_users, 0) AS churned_users,
  COALESCE(n.new_users, 0) - COALESCE(c.churned_users, 0) AS net_user_change,
  s.at_risk_users,
  s.dormant_users,
  s.cumulative_users,
  ROUND(COALESCE(c.churned_users, 0) * 100.0 / NULLIF(LAG(a.active_users) OVER (ORDER BY a.month_start), 0), 1) AS monthly_churn_rate_pct,
  a.month_start = CAST(DATE_TRUNC('MONTH', current_date()) AS DATE) AS is_partial_month
FROM active a
LEFT JOIN new_u n   USING (month_start)
LEFT JOIN churned c USING (month_start)
LEFT JOIN state s   USING (month_start)
ORDER BY a.month_start
""")
display(spark.table("gold_monthly_user_health"))

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold_cohort_retention AS
WITH um AS (
  SELECT CAST(DATE_TRUNC('MONTH', created_at) AS DATE) AS activity_month, user_email
  FROM raw_genie_messages WHERE user_email IS NOT NULL GROUP BY ALL
),
first_month AS (SELECT user_email, MIN(activity_month) AS cohort_month FROM um GROUP BY 1),
cohort_size AS (SELECT cohort_month, COUNT(*) AS cohort_users FROM first_month GROUP BY 1)
SELECT
  fm.cohort_month,
  cs.cohort_users,
  um.activity_month,
  CAST(MONTHS_BETWEEN(um.activity_month, fm.cohort_month) AS INT) AS months_since_join,
  COUNT(DISTINCT um.user_email) AS retained_users,
  ROUND(COUNT(DISTINCT um.user_email) * 100.0 / cs.cohort_users, 1) AS retention_pct
FROM first_month fm
JOIN um ON fm.user_email = um.user_email
JOIN cohort_size cs ON fm.cohort_month = cs.cohort_month
GROUP BY fm.cohort_month, cs.cohort_users, um.activity_month
""")
display(spark.sql("SELECT * FROM gold_cohort_retention ORDER BY cohort_month, months_since_join"))

In [0]:
# Adds behavioural features + AI failure reason / sentiment. Incremental: AI functions only run on NEW problem messages.
spark.sql("""
CREATE TABLE IF NOT EXISTS gold_enriched_messages (
  message_id STRING, conversation_id STRING, space_id STRING, space_name STRING, user_email STRING,
  created_at TIMESTAMP, question STRING, status STRING, genie_response STRING, response_seconds DOUBLE,
  feedback_rating STRING, feedback_comment STRING, source STRING,
  is_rephrase BOOLEAN, response_bucket STRING, is_problem BOOLEAN,
  failure_reason STRING, comment_sentiment STRING, enriched_at TIMESTAMP
)
""")

# Features computed over the full raw window (LAG needs conversation context)
spark.sql("""
CREATE OR REPLACE TEMP VIEW message_features AS
WITH m AS (
  SELECT * FROM raw_genie_messages WHERE user_email IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (PARTITION BY message_id ORDER BY created_at DESC) = 1
),
f AS (
  SELECT m.*, s.space_name,
         LAG(m.question) OVER (PARTITION BY m.conversation_id ORDER BY m.created_at) AS prev_question
  FROM m LEFT JOIN dim_genie_spaces s USING (space_id)
)
SELECT
  message_id, conversation_id, space_id, COALESCE(space_name, space_id) AS space_name, user_email, created_at,
  question, status, genie_response, response_seconds, feedback_rating, feedback_comment, source,
  COALESCE(prev_question IS NOT NULL AND question IS NOT NULL AND (
      question LIKE '%(rephrased)%'
      OR levenshtein(lower(question), lower(prev_question)) <= 0.5 * GREATEST(length(question), length(prev_question))
  ), FALSE) AS is_rephrase,
  CASE WHEN response_seconds IS NULL THEN 'Unknown'
       WHEN response_seconds <= 15 THEN 'Fast (<15s)'
       WHEN response_seconds <= 60 THEN 'Normal (15-60s)'
       ELSE 'Slow (>60s)' END AS response_bucket,
  -- COALESCE each term: a NULL (e.g. no feedback) must mean FALSE, not NULL
  (COALESCE(status IN ('FAILED', 'CANCELLED'), FALSE)
   OR COALESCE(feedback_rating = 'NEGATIVE', FALSE)
   OR COALESCE(response_seconds > 60, FALSE)
   OR COALESCE(genie_response RLIKE '(?i)(could not find|couldn.t find|no table|clarify|unable to|error|not available|does not contain|only goes up to)', FALSE)
  ) AS is_problem
FROM f
""")

# 1) Sync existing rows: late feedback updates; drop rows that left the raw window
spark.sql("""
MERGE INTO gold_enriched_messages t
USING message_features s ON t.message_id = s.message_id
WHEN MATCHED AND NOT (t.feedback_rating <=> s.feedback_rating AND t.feedback_comment <=> s.feedback_comment) THEN
  UPDATE SET t.feedback_rating = s.feedback_rating, t.feedback_comment = s.feedback_comment
WHEN NOT MATCHED BY SOURCE THEN DELETE
""")

# 2) Insert new messages; AI functions are evaluated only for new problem rows that have text
spark.sql("""
INSERT INTO gold_enriched_messages
SELECT
  f.message_id, f.conversation_id, f.space_id, f.space_name, f.user_email, f.created_at, f.question, f.status,
  f.genie_response, f.response_seconds, f.feedback_rating, f.feedback_comment, f.source,
  f.is_rephrase, f.response_bucket, f.is_problem,
  CASE
    WHEN NOT COALESCE(f.is_problem, FALSE) THEN NULL
    WHEN f.question IS NULL AND f.genie_response IS NULL THEN
      CASE WHEN f.response_seconds > 60 THEN 'timeout_or_slow'
           WHEN f.feedback_rating = 'NEGATIVE' THEN 'negative_feedback_no_text'
           ELSE 'failed_no_detail' END
    WHEN f.status = 'COMPLETED' AND f.response_seconds > 60 AND COALESCE(f.feedback_rating, '') <> 'NEGATIVE'
         AND NOT COALESCE(f.genie_response RLIKE '(?i)(could not find|couldn.t find|no table|clarify|unable to|error|not available|does not contain|only goes up to)', FALSE)
      THEN 'timeout_or_slow'
    ELSE ai_classify(
      concat('Question: ', COALESCE(f.question, ''), '\nGenie response: ', COALESCE(f.genie_response, ''),
             '\nStatus: ', COALESCE(f.status, ''), '\nUser feedback: ', COALESCE(f.feedback_rating, 'none'), ' ', COALESCE(f.feedback_comment, '')),
      ARRAY('query_error', 'timeout_or_slow', 'missing_data', 'missing_column_or_metric', 'ambiguous_question', 'wrong_answer', 'permission_denied', 'other'))
  END AS failure_reason,
  CASE WHEN f.feedback_comment IS NOT NULL THEN ai_analyze_sentiment(f.feedback_comment) END AS comment_sentiment,
  current_timestamp() AS enriched_at
FROM message_features f
LEFT ANTI JOIN gold_enriched_messages t ON t.message_id = f.message_id
""")
display(spark.sql("SELECT failure_reason, COUNT(*) n FROM gold_enriched_messages WHERE failure_reason IS NOT NULL GROUP BY 1 ORDER BY n DESC"))

In [0]:
# Churned vs at-risk vs active: per-user rates averaged by lifecycle status
spark.sql("""
CREATE OR REPLACE TABLE gold_churn_driver_summary AS
WITH per_user AS (
  SELECT e.user_email, l.lifecycle_status,
         COUNT(*) AS questions,
         AVG(CASE WHEN e.status IN ('FAILED','CANCELLED') THEN 1.0 ELSE 0 END) AS fail_rate,
         AVG(CAST(e.is_rephrase AS INT))  AS rephrase_rate,
         AVG(CASE WHEN e.response_seconds > 60 THEN 1.0 ELSE 0 END) AS slow_rate,
         AVG(CASE WHEN e.feedback_rating = 'NEGATIVE' THEN 1.0 ELSE 0 END) AS neg_rate,
         AVG(CAST(e.is_problem AS INT))   AS problem_rate,
         AVG(e.response_seconds)          AS avg_secs
  FROM gold_enriched_messages e JOIN gold_user_lifecycle l USING (user_email)
  GROUP BY 1, 2
)
SELECT lifecycle_status, COUNT(*) AS users,
       ROUND(AVG(questions), 1)          AS avg_questions,
       ROUND(AVG(fail_rate) * 100, 1)    AS avg_failure_pct,
       ROUND(AVG(rephrase_rate) * 100, 1) AS avg_rephrase_pct,
       ROUND(AVG(slow_rate) * 100, 1)    AS avg_slow_pct,
       ROUND(AVG(neg_rate) * 100, 1)     AS avg_negative_feedback_pct,
       ROUND(AVG(problem_rate) * 100, 1) AS avg_problem_pct,
       ROUND(AVG(avg_secs), 1)           AS avg_response_seconds
FROM per_user GROUP BY 1
""")

spark.sql("""
CREATE OR REPLACE TABLE gold_failure_reasons AS
SELECT e.space_id, e.space_name, e.failure_reason, l.lifecycle_status,
       COUNT(*) AS occurrences, COUNT(DISTINCT e.user_email) AS affected_users
FROM gold_enriched_messages e JOIN gold_user_lifecycle l USING (user_email)
WHERE e.failure_reason IS NOT NULL
GROUP BY ALL
""")

spark.sql("""
CREATE OR REPLACE TABLE gold_space_health AS
WITH s AS (
  SELECT e.*, l.lifecycle_status FROM gold_enriched_messages e JOIN gold_user_lifecycle l USING (user_email)
),
agg AS (
  SELECT space_id, ANY_VALUE(space_name) AS space_name,
         COUNT(DISTINCT user_email) AS users,
         COUNT(DISTINCT CASE WHEN lifecycle_status = 'Active'  THEN user_email END) AS active_users,
         COUNT(DISTINCT CASE WHEN lifecycle_status = 'At Risk' THEN user_email END) AS at_risk_users,
         COUNT(DISTINCT CASE WHEN lifecycle_status = 'Churned' THEN user_email END) AS churned_users,
         COUNT(*) AS questions,
         ROUND(AVG(CASE WHEN status IN ('FAILED','CANCELLED') THEN 1.0 ELSE 0 END) * 100, 1) AS failure_pct,
         ROUND(COUNT_IF(feedback_rating = 'NEGATIVE') * 100.0 / NULLIF(COUNT_IF(feedback_rating IS NOT NULL), 0), 1) AS negative_feedback_pct,
         ROUND(AVG(response_seconds), 1) AS avg_response_seconds,
         ROUND(AVG(CAST(is_rephrase AS INT)) * 100, 1) AS rephrase_pct,
         ROUND(AVG(CAST(is_problem AS INT)) * 100, 1) AS problem_pct
  FROM s GROUP BY space_id
),
top AS (
  SELECT space_id, MAX_BY(failure_reason, n) AS top_failure_reason
  FROM (SELECT space_id, failure_reason, COUNT(*) n FROM s WHERE failure_reason IS NOT NULL GROUP BY 1, 2)
  GROUP BY 1
)
SELECT agg.*, ROUND(churned_users * 100.0 / NULLIF(users, 0), 1) AS churned_user_pct, top.top_failure_reason
FROM agg LEFT JOIN top USING (space_id)
""")
display(spark.table("gold_churn_driver_summary"))
display(spark.sql("SELECT * FROM gold_space_health ORDER BY churned_users DESC"))

In [0]:
# Primary churn reason = most frequent failure reason in the user's last 14 active days.
# churn_risk_score (0-100) = inactivity (40) + recent problem rate (30) + usage decline (15) + recent negative feedback (15).
spark.sql(f"""
CREATE OR REPLACE TABLE gold_user_risk AS
WITH recent AS (
  SELECT e.user_email, e.failure_reason, e.is_problem, e.is_rephrase, e.feedback_rating
  FROM gold_enriched_messages e JOIN gold_user_lifecycle l USING (user_email)
  WHERE DATE(e.created_at) >= l.last_question_date - INTERVAL 14 DAYS
),
top_reason AS (
  SELECT user_email, MAX_BY(failure_reason, n) AS primary_reason
  FROM (SELECT user_email, failure_reason, COUNT(*) n FROM recent WHERE failure_reason IS NOT NULL GROUP BY 1, 2)
  GROUP BY 1
),
recent_stats AS (
  SELECT user_email, COUNT(*) AS recent_questions,
         AVG(CAST(is_problem AS INT)) AS recent_problem_rate,
         AVG(CAST(is_rephrase AS INT)) AS recent_rephrase_rate,
         COUNT_IF(feedback_rating = 'NEGATIVE') AS recent_negative_feedback
  FROM recent GROUP BY 1
)
SELECT
  l.user_email, l.department, l.role, l.lifecycle_status, l.is_new_user,
  l.first_question_date, l.last_question_date, l.days_since_last_question,
  l.total_questions, l.questions_last_28d, l.questions_prev_28d,
  COALESCE(s.space_name, l.last_space_id) AS last_space_name,
  rs.recent_questions,
  ROUND(rs.recent_problem_rate * 100, 1)  AS recent_problem_pct,
  ROUND(rs.recent_rephrase_rate * 100, 1) AS recent_rephrase_pct,
  rs.recent_negative_feedback,
  COALESCE(tr.primary_reason, CASE WHEN l.total_questions <= 2 THEN 'low_engagement_trial' ELSE 'no_clear_failure' END) AS primary_churn_reason,
  ROUND(LEAST(100,
      40 * LEAST(1, l.days_since_last_question / {CHURN}.0)
    + 30 * COALESCE(rs.recent_problem_rate, 0)
    + 15 * CASE WHEN l.questions_prev_28d > 0 THEN GREATEST(0, 1 - l.questions_last_28d / l.questions_prev_28d) ELSE 0 END
    + 15 * LEAST(1, COALESCE(rs.recent_negative_feedback, 0) / 3.0)
  ), 1) AS churn_risk_score
FROM gold_user_lifecycle l
LEFT JOIN recent_stats rs USING (user_email)
LEFT JOIN top_reason tr   USING (user_email)
LEFT JOIN dim_genie_spaces s ON s.space_id = l.last_space_id
""")
display(spark.sql("""SELECT lifecycle_status, primary_churn_reason, COUNT(*) users, ROUND(AVG(churn_risk_score),1) avg_risk
                     FROM gold_user_risk GROUP BY 1, 2 ORDER BY 1, 3 DESC"""))

In [0]:
# Set grant_to_principal to the app's service principal application ID (or a group) so the app can read gold tables.
if GRANTEE:
    for stmt in [f"GRANT USE CATALOG ON CATALOG `{CATALOG}` TO `{GRANTEE}`",
                 f"GRANT USE SCHEMA, SELECT ON SCHEMA `{CATALOG}`.`{SCHEMA}` TO `{GRANTEE}`"]:
        spark.sql(stmt)
        print(stmt)
else:
    print("grant_to_principal not set - skipping grants")